In [0]:
#Verify if ingestion landed correctly
dbutils.fs.ls("/Volumes/adb_healthcare/landing/landing_volume/neon_postgres/hospitals/2026-10-22_12:36:14/")

In [0]:
# Create widgets for parameters
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")
# Optional: target bronze schema. Defaults to 'bronze' so ADF runs are unchanged; use e.g. 'bronze_test' for verification runs
dbutils.widgets.text("target_schema", "bronze", "Target Schema")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")
target_schema = dbutils.widgets.get("target_schema") or "bronze"
print(f"Params: source_system={source_system}, table_name={table_name}, landing_timestamp={landing_timestamp}, source_file_format={source_file_format}, target_schema={target_schema}")


In [0]:
import os

landing_path = f"/Volumes/adb_healthcare/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"
print(f"Checking landing path: {landing_path}")

# ADF's Copy only copies source files modified since the last run, so a run with no new files creates no folder.
# In that case there is nothing to load: end the notebook successfully without touching bronze.
landing_files = [f for f in os.listdir(landing_path) if not f.startswith(('_', '.'))] if os.path.isdir(landing_path) else []
if not landing_files:
    print(f"[SKIP] No files found in {landing_path} - no new source data for {table_name}. Bronze left unchanged.")
    dbutils.notebook.exit("0")
print(f"Found {len(landing_files)} file(s): {landing_files[:5]}")

#read file based on source file format
if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(landing_path)
else:
    landing_df = spark.read.format('parquet').load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = (landing_df.withColumn("landing_timestamp", lit(landing_timestamp))
                        .withColumn("insert_timestamp", current_timestamp()))

In [0]:
#write to bronze table
target_table = f"adb_healthcare.{target_schema}.{table_name}"
print(f"Appending to {target_table}")
updated_df.write.mode('append').saveAsTable(target_table)
batch_rows = spark.table(target_table).filter(f"landing_timestamp = '{landing_timestamp}'").count()
print(f"Done: {target_table} now has {batch_rows} rows for landing_timestamp={landing_timestamp}")